In [1]:
platformID = 'FBE'


In [2]:
from datetime import datetime
import pandas as pd

import os 

## import helper 

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules
from functions import lookup_loader, execute_sql_query, calculate_rolling_avg_country_split, apply_first_split_backfill, compare_or_update_reference, fix_country_one_percent_prev_week
import test_functions

from config import gam_info

In [4]:
lookup = lookup_loader(gam_info, platformID, '1c',
                       with_country=True, country_col=['YT-_FBE_codes'])
week_tester = lookup['week_tester']
socialmedia_accounts = lookup['socialmedia_accounts']
country_codes = lookup['country_codes']
socialmedia_accounts.head(10)



✅ Test FBE_1c_00 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1c_01 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1c_02 passed: No empty values in lookup.
...updating logbook...

✅ Test FBE_1c_03 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1c_04 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1c_05 passed: No empty values in lookup.
...updating logbook...

✅ Test FBE_1c_06 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test FBE_1c_07 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test FBE_1c_08 passed: No empty values in lookup.
...updating logbook...



,PlatformID,Status,Comment,Channel ID,Channel Name,Service,ServiceID,Excluding UK,Channel URL,Channel Username,Linked FB Account,Start,End
3,FBE,active,NaN,FBE100163990128209,BBC Dari,Dari,DAR,NaN,NaN,bbcdari,NaN,2020-04-01,NaT
4,FBE,active,NaN,FBE100826406677543,BBC Springwatch,Studios,WOR,Yes,NaN,BBCSpringwatch,NaN,2020-04-01,NaT
5,FBE,active,NaN,FBE101242425117794,Morning Live,Studios,WOR,Yes,NaN,bbcmorninglive,NaN,2020-04-01,NaT
6,FBE,active,NaN,FBE10150118096995434,BBC News Indonesia,Indonesian,INO,NaN,NaN,BBCNewsIndonesia,NaN,2020-04-01,NaT
9,FBE,active,NaN,FBE102186576502930,BBC Persian TV,Persian,PER,NaN,NaN,bbcpersiantv,NaN,2020-04-01,NaT
10,FBE,active,barely active,FBE102775241582303,Mastermind Cymru,Studios,WOR,Yes,NaN,mastermindcymru,NaN,2020-04-01,NaT
11,FBE,active,NaN,FBE103678496456574,BBC News සිංහල,Sinhala,SIN,NaN,NaN,BBCnewsSinhala,NaN,2020-04-01,NaT
12,FBE,active,NaN,FBE103894041188535,DIY SOS,Studios,WOR,Yes,NaN,DIYSOSofficial,NaN,2020-04-01,NaT
14,FBE,active,NaN,FBE1048672825173961,Antiques Roadshow,Studios,WOR,Yes,NaN,BBCAntiquesRoadshow,NaN,2020-04-01,NaT
18,FBE,active,NaN,FBE1068750829805728,BBC Uzbek Afghanistan,Uzbek,UZB,NaN,NaN,bbcnewsuzbekafghan,NaN,2020-04-01,NaT


## country

In [5]:
sql_query = f"""
    SELECT 
        week_commencing,
        page_id ,
        page_name,
        page_fans_country_total AS page_fans_country,
        country_code AS fb_metric_breakdown
    FROM
        redshiftdb.central_insights.adverity_social_facebook_page_fans_by_country
    WHERE
        week_commencing Between '{gam_info['w/c_start']}' and '{gam_info['w/c_end']}'
        ;
    """
file = f"../data/raw/{platformID}/{gam_info['file_timeinfo']}_{platformID}_country_redshift_extract.csv"

try: 
    df = execute_sql_query(sql_query)
    df['page_id'] = platformID+df['page_id']
    df.to_csv(file, index=False, na_rep='')
except Exception as e:
    print("No Redshift connection or another error occurred. Using last time queried.")
    print(f"Error: {e}")

facebook_country_raw = pd.read_csv(file, keep_default_na=False, dtype={"page_id": "string"}).drop_duplicates()

# if not facebook_country_raw['page_id'].astype(str).str.startswith('FBE') :
#     facebook_country_raw['page_id'] = platformID+facebook_country_raw['page_id']
#     print("--in if condition--")

facebook_country_raw['week_commencing'] = pd.to_datetime(facebook_country_raw['week_commencing'])
facebook_country_raw = facebook_country_raw.rename(columns={'page_id': 'Channel ID',
                                                            'page_name': 'Channel Name',
                                                            'week_commencing': 'w/c',
                                                            'fb_metric_breakdown': 'YT-_FBE_codes'})

In [6]:
facebook_country_raw

,w/c,Channel ID,Channel Name,page_fans_country,YT-_FBE_codes
0,2025-03-31,FBE15286229625,BBC Yorkshire,1405,FR
1,2025-03-31,FBE23304592572,BBC Symphony Orchestra,2538,DE
2,2025-03-31,FBE122238607940771,Travelling Folk on BBC Radio Scotland,14,BR
3,2025-03-31,FBE172228666137775,BBC Philharmonic,1541,AU
4,2025-03-31,FBE173825495996416,BBC Brit,59,IE
...,...,...,...,...,...
577879,2026-01-05,FBE151955124848859,BBC News India,106021,US
577880,2026-01-05,FBE463402774003391,BBC News Telugu,573,TH
577881,2026-01-05,FBE107629172665153,BBC Cumbria,427,NZ
577882,2026-01-05,FBE104125254872671,BBC Bradford,4,AE


In [7]:
channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()

### RUN TESTS
# missing page_ids
test_functions.test_filter_elements_returned(facebook_country_raw, 
                                             channel_ids, 
                                             'Channel ID', 
                                             f"{platformID}_1c_09",
                                             "Check that all page IDs are found in SQL")


# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               channel_id_col=['Channel ID'],
                                               main_data=facebook_country_raw,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=socialmedia_accounts[['Channel ID', 'Start', 'End']],
                                               test_number=f"{platformID}_1c_10",
                                               test_step="Check all weeks present for each account")

# missing values per week / page id 
test_functions.test_non_null_and_positive(facebook_country_raw, 
                           numeric_columns=['page_fans_country'], 
                           test_number=f"{platformID}_1c_11",
                           test_step='Check no missing values in page fans column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(facebook_country_raw, ['Channel ID', 'w/c', 'YT-_FBE_codes'], 
                               test_number=f"{platformID}_1c_12",
                               test_step='Check no duplicates from redshift returned')

...testing Channel ID...
✅ Test FBE_1c_09 passed: everything found!
...updating logbook...

❌ Missing weeks from Start onward:
               Channel ID      Start End        w/c
3740   FBE630866223444617 2025-06-23 NaT 2025-06-23
1259   FBE151173098230967 2020-04-01 NaT 2025-06-23
3741   FBE630866223444617 2025-06-23 NaT 2025-06-30
14     FBE100163990128209 2020-04-01 NaT 2025-07-07
2421      FBE230299653821 2020-04-01 NaT 2025-07-07
...                   ...        ...  ..        ...
730    FBE124158667615790 2020-04-01 NaT 2026-01-19
2062  FBE1801602293398610 2020-04-01 NaT 2026-01-19
2750      FBE282681245570 2020-04-01 NaT 2026-01-19
257    FBE102775241582303 2020-04-01 NaT 2026-01-19
1375   FBE147105585468223 2020-04-01 NaT 2026-01-19

[729 rows x 4 columns]

Summary of missing weeks per channel_id_col:
               Channel ID  missing_week_count
43     FBE171824429536304                  25
17     FBE124158667615790                  13
79     FBE359687864111179                

In [8]:
# filter to relevant channel ids
facebook_country = facebook_country_raw[facebook_country_raw['Channel ID'].isin(channel_ids)]

print(facebook_country.shape)

# fill missing countries 
facebook_country['YT-_FBE_codes'] = facebook_country['YT-_FBE_codes'].replace('', 'ZZ')

print(facebook_country.shape)

# filter to relevant countries
test_functions.test_inner_join(facebook_country, 
                               country_codes, 
                               ['YT-_FBE_codes'], 
                               f"{platformID}_1c_13",
                               test_step='calculating country %',
                                focus='left')

facebook_country = facebook_country.merge(country_codes[['YT-_FBE_codes', 'PlaceID']], 
                                          on='YT-_FBE_codes', 
                                          how='left').drop(columns=['YT-_FBE_codes'])

print(facebook_country.shape)


(163787, 5)
(163787, 5)
✅ Inner join test FBE_1c_13 successful: No issues found.
...updating logbook...

(163787, 5)


/var/folders/k_/88tnxx056sn1mg7_z9w333l40000gn/T/ipykernel_90178/2937215015.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  facebook_country['YT-_FBE_codes'] = facebook_country['YT-_FBE_codes'].replace('', 'ZZ')


In [9]:

# Group by specified columns and sum the fb_metric_value
facebook_country_sum = facebook_country.groupby(['Channel ID', 'w/c'])\
                                        .agg(Sum_page_fans_country=('page_fans_country', 'sum'))\
                                        .reset_index()
facebook_country = facebook_country.merge(facebook_country_sum, 
                                          how='inner', on= ['Channel ID', 'w/c'])
test_functions.test_inner_join(facebook_country, facebook_country_sum, 
                               ['Channel ID', 'w/c'], 
                               f"{platformID}_1c_14", 
                               test_step='calculating country %')

facebook_country['country_%'] = facebook_country['page_fans_country']/facebook_country['Sum_page_fans_country']

### RUN TESTS
test_functions.test_percentage(facebook_country, 
                               ['Channel ID', 'w/c'], 
                               f"{platformID}_1c_15", 
                               test_step='summing country % per week & account')

✅ Inner join test FBE_1c_14 successful: No issues found.
...updating logbook...

...updating logbook...



In [10]:
facebook_country

,w/c,Channel ID,Channel Name,page_fans_country,PlaceID,Sum_page_fans_country,country_%
0,2025-03-31,FBE173825495996416,BBC Brit,59,IRE,168163,0.000351
1,2025-03-31,FBE264572343581678,BBC News বাংলা,33713,BUR,14493252,0.002326
2,2025-03-31,FBE10150118096995434,BBC News Indonesia,1691760,INO,2036080,0.830891
3,2025-03-31,FBE485274381864409,BBC News Amharic,626,AFG,822986,0.000761
4,2025-03-31,FBE264572343581678,BBC News বাংলা,21188,CAN,14493252,0.001462
...,...,...,...,...,...,...,...
163782,2026-01-05,FBE1068750829805728,BBC Uzbek Afghanistan,498,OMA,644939,0.000772
163783,2026-01-05,FBE422868177848193,BBC Holby City,306,JER,232264,0.001317
163784,2026-01-05,FBE109993787372855,Natwampane,7,BEN,17277,0.000405
163785,2026-01-05,FBE151955124848859,BBC News India,106021,USA,4055912,0.026140


In [11]:
# calculate rolling average for missing weeks ?
avg_country_df = calculate_rolling_avg_country_split(facebook_country, 'country_%', 
                                                     week_tester['w/c'].min(), week_tester['w/c'].max())

if len(avg_country_df) > 0:
# new channels have missing country splits for the first few weeks
    avg_backfill_country_df = apply_first_split_backfill(avg_country_df, 
                                                          socialmedia_accounts, 
                                                          week_tester
                                                         )
else:
    avg_backfill_country_df = pd.DataFrame(columns=['Channel ID', 'w/c'])


/Users/nallas02/BBC Dropbox/Srimayi Nallanchakravarthula/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/functions.py:422: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.assign(**{value_col: g[value_col].ffill().bfill()}))


In [13]:
# Canonical channel list and canonical week list
channel_list = facebook_country[['Channel ID']].drop_duplicates()
week_list = week_tester[['w/c']].drop_duplicates()

# Build full (Channel × Week) grid via cross join
expected_grid = channel_list.merge(week_list, how='cross')

# Join channel start/end info and keep only weeks within each channel's active window
# - Include weeks on/after Start
# - Include weeks on/before End (or 'today' if End is missing)
expected_grid = expected_grid.merge(
    socialmedia_accounts[['Channel ID', 'Start', 'End']],
    on='Channel ID',
    how='left'
)

today = pd.Timestamp.today().normalize()
expected_grid = expected_grid[
    (expected_grid['Start'] <= expected_grid['w/c']) &
    (expected_grid['End'].fillna(today) >= expected_grid['w/c'])
]

# Left-join actual engagements to the expected grid to detect missing rows
filled_grid = expected_grid.merge(
    facebook_country,
    on=['Channel ID', 'w/c'],
    how='left',
    indicator=True
)

# Mark rows that are missing in the original data
filled_grid['missing_week'] = (filled_grid['_merge'] == 'left_only')
filled_grid = filled_grid.drop(columns=['_merge'])

# Join per-channel rolling averages (already computed) to supply fill values
# Suffix `_avg` ensures we can distinguish average columns from originals
if len(avg_backfill_country_df) > 0 :
    result_df = filled_grid.merge(
        avg_backfill_country_df,
        on=['Channel ID', 'w/c'],
        how='left',
        suffixes=['', '_avg']
    ).drop_duplicates()

    # Fill NaNs in original columns with per-channel averages (only where original is missing)
    result_df['PlaceID']   = result_df['PlaceID'].fillna(result_df['PlaceID_avg'])
    result_df['country_%'] = result_df['country_%'].fillna(result_df['country_%_avg'])
else:
    result_df = filled_grid    


# duplicates because there is a row expansion between main dataset and average for weeks that have country data
cols = ['Channel ID', 'Channel Name', 'w/c', 'PlaceID', 'country_%']
result_df = result_df[cols].drop_duplicates().dropna(subset='country_%')
# result_df now contains:
# - All (Channel ID, w/c) rows within each channel's Start–End window
# - `missing_week=True` for synthetic rows added from the expected grid
# - Filled `PlaceID` / `country_%` from per-channel averages where originals were NaN


In [15]:

# Iran = PlaceID "IRN"
IRAN_CODE = "IRN"

result_df = fix_country_one_percent_prev_week(
    result_df,
    IRAN_CODE,
    ["2026-01-19","2026-01-12"]
)

Fixing Iran reach to 1% of previous week's value for weeks: ['2026-01-19', '2026-01-12']
Affected rows: (9805, 5)


/Users/nallas02/BBC Dropbox/Srimayi Nallanchakravarthula/BBC - InfoLab - Lumen 2020/digiGAM_ytd/digiGAM/helper/functions.py:190: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  affected = df[df['w/c'].isin(set(week_list))]


In [16]:
#sanity checks 
# 1
#display(result_df[(result_df['w/c'] == '2026-01-12') & (result_df['PlaceID'] == 'IRN')].head())
# 2
#display(result_df[(result_df['w/c'] == '2026-01-12') & (result_df['Channel ID'] == 'FBE102186576502930')].sort_values('PlaceID'))
# 3 
print(result_df[(result_df['w/c'] == '2026-01-12') & (result_df['Channel ID'] == 'FBE102186576502930')].sort_values('PlaceID')['country_%'].sum())

#sanity checks - previous weeks are not affected
# 4
#display(result_df[(result_df['w/c'] == '2026-01-05') & (result_df['PlaceID'] == 'IRN')].head())
# 5
#display(result_df[(result_df['w/c'] == '2026-01-05') & (result_df['Channel ID'] == 'FBE102186576502930')].sort_values('PlaceID'))
# 6 
print(result_df[(result_df['w/c'] == '2026-01-05') & (result_df['Channel ID'] == 'FBE102186576502930')].sort_values('PlaceID')['country_%'].sum())

1.002369872177607
1.001293690488409


In [17]:
# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               channel_id_col=['Channel ID'],
                                               main_data=result_df,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=socialmedia_accounts[['Channel ID', 'Start', 'End']],
                                               test_number=f"{platformID}_1c_16",
                                               test_step="After rolling average: Check all weeks present for each account")

# missing values per week / page id 
test_functions.test_non_null_and_positive(result_df, 
                           numeric_columns=['country_%'], 
                           test_number=f"{platformID}_1c_17",
                           test_step='After rolling average: Check no missing values in page fans column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(result_df, 
                               ['Channel ID', 'w/c', 'PlaceID'], 
                               test_number=f"{platformID}_1c_18",
                               test_step='After rolling average: Check no duplicates from redshift returned')

✅ No missing weeks from each channel’s Start onward.
...updating logbook...

✅ Test FBE_1c_17 passed: No NaN and all numeric values > 0.
...updating logbook...

✅ Test FBE_1c_18 passed: No combinations occurs more than once a week.
...updating logbook...



In [18]:
file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

cols = ['Channel ID', 'Channel Name', 'w/c', 'PlaceID', 'country_%']
result_df[cols].to_csv(f"{file_path}/{gam_info['file_timeinfo']}_{platformID}_REDSHIFT_COUNTRY.csv",
                        index=None)

In [19]:
result_df

,Channel ID,Channel Name,w/c,PlaceID,country_%
4196186,FBE100163990128209,BBC Dari,2025-03-31,THA,0.000632
4196232,FBE100163990128209,BBC Dari,2025-03-31,TAD,0.002266
4196278,FBE100163990128209,BBC Dari,2025-03-31,OST,0.006050
4196324,FBE100163990128209,BBC Dari,2025-03-31,BUR,0.003545
4196370,FBE100163990128209,BBC Dari,2025-03-31,KUW,0.001489
...,...,...,...,...,...
4558890,FBE948946275170651,bbcglobalwomen,2026-01-19,TAI,0.004158
4558938,FBE948946275170651,bbcglobalwomen,2026-01-19,TAN,0.005255
4558986,FBE948946275170651,bbcglobalwomen,2026-01-19,USA,0.115512
4559034,FBE948946275170651,bbcglobalwomen,2026-01-19,VIE,0.007842
